In [9]:
# Install required library
!pip install torch torchvision torchaudio

import torch
import torch.nn as nn
import torch.optim as optim

# --------------------------
# Sample Dataset
# --------------------------
pairs = [
    ("hello", "hi how are you"),
    ("how are you", "i am fine"),
    ("what is your name", "my name is chatbot"),
    ("bye", "goodbye")
]

# --------------------------
# Vocabulary Building
# --------------------------
word2idx = {"<pad>":0, "<sos>":1, "<eos>":2}
idx2word = {0:"<pad>", 1:"<sos>", 2:"<eos>"}

def add_words(sentence):
    for word in sentence.split():
        if word not in word2idx:
            idx = len(word2idx)
            word2idx[word] = idx
            idx2word[idx] = word

for inp, out in pairs:
    add_words(inp)
    add_words(out)

vocab_size = len(word2idx)

# --------------------------
# Convert sentence to tensor
# --------------------------
def sentence_to_tensor(sentence):
    tokens = [word2idx[word] for word in sentence.split()]
    return torch.tensor(tokens, dtype=torch.long)

# --------------------------
# Attention Model
# --------------------------
class AttentionChatbot(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

        self.attention = nn.Linear(hidden_size, 1)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embed = self.embedding(x)

        output, (hidden, cell) = self.lstm(embed)

        # Attention weights
        attn_weights = torch.softmax(self.attention(output), dim=1)

        # Context vector
        context = torch.sum(attn_weights * output, dim=1)

        out = self.fc(context)

        return out, attn_weights

# --------------------------
# Model Setup
# --------------------------
embed_size = 16
hidden_size = 32

model = AttentionChatbot(vocab_size, embed_size, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# --------------------------
# Training
# --------------------------
for epoch in range(300):

    total_loss = 0

    for inp, out in pairs:
        input_tensor = sentence_to_tensor(inp).unsqueeze(0)

        target_word = out.split()[0]   # first output word only
        target_tensor = torch.tensor([word2idx[target_word]])

        optimizer.zero_grad()

        output, attn = model(input_tensor)

        loss = criterion(output, target_tensor)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# --------------------------
# Testing
# --------------------------
test = "how are you"
test_tensor = sentence_to_tensor(test).unsqueeze(0)

output, attn = model(test_tensor)

predicted = torch.argmax(output, dim=1).item()

print("\nInput:", test)
print("Predicted Output:", idx2word[predicted])

# --------------------------
# Attention Weights
# --------------------------
print("Attention Weights:")
print(attn.squeeze().detach().numpy())

Epoch 0, Loss: 11.7285
Epoch 50, Loss: 0.0107
Epoch 100, Loss: 0.0041
Epoch 150, Loss: 0.0022
Epoch 200, Loss: 0.0014
Epoch 250, Loss: 0.0010

Input: how are you
Predicted Output: i
Attention Weights:
[0.01214238 0.16882874 0.8190289 ]
